# Transformer-Based Multi-Modal Document Intelligence Pipeline

This notebook implements an end-to-end LayoutLMv3-based document understanding pipeline for intelligent insurance document extraction and analysis. It covers dataset preprocessing, document filtering, OCR-aware token and bounding-box processing, data augmentation, transformer fine-tuning, validation workflows, evaluation metrics generation, and structured field extraction for insurance-related documents such as hospital bills and pharmacy bills using PaddleOCR and LayoutLMv3. **bold text**

### Dataset Filtering & Organization

This merges annotated JSON datasets, filters hospital and pharmacy documents using filename-based categorization, and generates separate structured datasets for downstream LayoutLMv3 training workflows.

In [ ]:
import json

# ===== INPUT FILES =====
TRAIN_JSON = "/content/drive/MyDrive/SplittedOutputOfMedicliamSystem/train.json"
TEST_JSON  = "/content/drive/MyDrive/SplittedOutputOfMedicliamSystem/test.json"

OUTPUT_DIR = "/content/drive/MyDrive/SplittedOutputOfMedicliamSystem"

TRAIN_HOSPITAL = OUTPUT_DIR + "/train_hospital.json"
TEST_PHARMACY  = OUTPUT_DIR + "/test_pharmacy.json"

# ===== LOAD BOTH FILES =====
with open(TRAIN_JSON) as f:
    train_data = json.load(f)

with open(TEST_JSON) as f:
    test_data = json.load(f)

# ===== MERGE DATA =====
all_data = train_data + test_data

# ===== SPLIT =====
train_hospital = []
test_pharmacy = []

for item in all_data:
    path = item["file_name"].lower()

    if "hospital" in path:
        train_hospital.append(item)
    elif "pharmacy" in path:
        test_pharmacy.append(item)

# ===== SAVE FILES =====
with open(TRAIN_HOSPITAL, "w") as f:
    json.dump(train_hospital, f, indent=2)

with open(TEST_PHARMACY, "w") as f:
    json.dump(test_pharmacy, f, indent=2)

# ===== PRINT INFO =====
print("Train (Hospital):", len(train_hospital))
print("Test (Pharmacy):", len(test_pharmacy))

Train (Hospital): 23
Test (Pharmacy): 29


### LayoutLMv3 Training & Augmentation Pipeline

This cell initializes the LayoutLMv3 processor, prepares augmented training datasets and validation dataloaders, fine-tunes the transformer model using PyTorch, applies optimization techniques such as gradient clipping, weight decay, learning-rate scheduling, and early stopping, evaluates model performance using validation metrics, and saves the best-performing model checkpoint.

In [ ]:


import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import (
    LayoutLMv3ImageProcessor,
    LayoutLMv3TokenizerFast,
    LayoutLMv3Processor,
    get_linear_schedule_with_warmup
)

from loader import LayoutLMv3Dataset
from trainer import ModelModule
from engine import eval_with_metrics

# ================= CONFIG =================

BASE_DIR = "/content/drive/MyDrive/SplittedOutputOfMedicliamSystem"

TRAIN_JSON = os.path.join(BASE_DIR, "train_hospital.json")
TEST_JSON  = os.path.join(BASE_DIR, "test_pharmacy.json")

MODEL_BASE = "microsoft/layoutlmv3-base"
SAVE_DIR = BASE_DIR

BATCH_SIZE = 2
EPOCHS = 30
LR = 5e-5
NUM_LABELS = 6
PATIENCE = 3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")




def main():

    # ---------- PROCESSOR ----------
    image_processor = LayoutLMv3ImageProcessor(apply_ocr=False)

    tokenizer = LayoutLMv3TokenizerFast.from_pretrained(
        MODEL_BASE,
        ignore_mismatched_sizes=True
    )

    processor = LayoutLMv3Processor(
        image_processor=image_processor,
        tokenizer=tokenizer
    )

    # ---------- DATASET ----------
    train_dataset = LayoutLMv3Dataset(
        TRAIN_JSON, processor, BASE_DIR, 256, augment=True   # 🔥 ON
    )

    test_dataset = LayoutLMv3Dataset(
        TEST_JSON, processor, BASE_DIR, 256, augment=False  # 🔥 OFF
    )

    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE)

    # ---------- MODEL ----------
    model = ModelModule(num_labels=NUM_LABELS, model_name_or_path=MODEL_BASE)
    model.to(DEVICE)

    # ---------- OPTIMIZER ----------
    optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)  # 🔥 weight decay

    # ---------- SCHEDULER ----------
    total_steps = len(train_loader) * EPOCHS

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=int(0.1 * total_steps),
        num_training_steps=total_steps
    )

    best_loss = float("inf")
    patience_counter = 0
    loss_history = []

    # ---------- TRAIN LOOP ----------
    for epoch in range(EPOCHS):

        print(f"\n🚀 Epoch {epoch+1}/{EPOCHS}")

        model.train()
        total_loss = 0

        for batch in train_loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}

            logits, loss = model(**batch)

            loss.backward()

            # 🔥 gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

            total_loss += loss.item()

        train_loss = total_loss / len(train_loader)
        loss_history.append(train_loss)

        print(f"Train Loss: {train_loss:.4f}")

        # ---------- EVALUATION ----------
        eval_metrics = eval_with_metrics(
            test_loader,
            model,
            DEVICE,
            NUM_LABELS
        )

        val_loss = eval_metrics["loss"]
        val_acc  = eval_metrics["accuracy"]

        print(f"Val Loss: {val_loss:.4f}")
        print(f"Val Acc: {val_acc:.4f}")

        # ---------- SAVE BEST MODEL ----------
        if val_loss < best_loss:
            best_loss = val_loss
            patience_counter = 0

            torch.save(
                model.state_dict(),
                os.path.join(SAVE_DIR, "best_model.bin")
            )
            print("✅ Best model saved")

        else:
            patience_counter += 1

        # ---------- EARLY STOPPING ----------
        if patience_counter >= PATIENCE:
            print("⛔ Early stopping triggered")
            break

    # ---------- SAVE LOSS HISTORY ----------
    np.save(
        os.path.join(SAVE_DIR, "loss_history.npy"),
        np.array(loss_history)
    )


if __name__ == "__main__":
    main()

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.weight                  | MISSING    | 
classifier.bias                    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



🚀 Epoch 1/30
Train Loss: 1.8552


100%|██████████| 15/15 [00:02<00:00,  7.00it/s]


Val Loss: 1.6603
Val Acc: 0.2875
✅ Best model saved

🚀 Epoch 2/30
Train Loss: 1.5072


...

Val Loss: 0.1271
Val Acc: 0.9688

🚀 Epoch 12/30
Train Loss: 0.0087


100%|██████████| 15/15 [00:02<00:00,  6.69it/s]


Val Loss: 0.2126
Val Acc: 0.9563

🚀 Epoch 13/30
Train Loss: 0.0043


100%|██████████| 15/15 [00:02<00:00,  6.41it/s]


Val Loss: 0.3508
Val Acc: 0.9250
⛔ Early stopping triggered


### LayoutLMv3 Evaluation & Metrics Pipeline

This cell loads the fine-tuned LayoutLMv3 model, performs inference on the validation dataset, filters token-level predictions, computes evaluation metrics including accuracy, precision, recall, and F1-score, and generates a detailed classification report for document field extraction labels such as HOSPITAL, PATIENT, DATE, ADDRESS, and AMOUNT.

In [ ]:
import os
import torch
import numpy as np
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

from loader import LayoutLMv3Dataset
from trainer import ModelModule
from transformers import LayoutLMv3ImageProcessor, LayoutLMv3TokenizerFast, LayoutLMv3Processor


BASE_DIR = "/content/drive/MyDrive/SplittedOutputOfMedicliamSystem"
TEST_JSON = os.path.join(BASE_DIR, "test_pharmacy.json")
MODEL_PATH = os.path.join(BASE_DIR, "best_model.bin")

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_LABELS = 6

# ===== PROCESSOR =====
image_processor = LayoutLMv3ImageProcessor(apply_ocr=False)
tokenizer = LayoutLMv3TokenizerFast.from_pretrained("microsoft/layoutlmv3-base")

processor = LayoutLMv3Processor(
    image_processor=image_processor,
    tokenizer=tokenizer
)

# ===== DATASET =====
test_dataset = LayoutLMv3Dataset(TEST_JSON, processor, BASE_DIR, 256)
test_loader = DataLoader(test_dataset, batch_size=2)

# ===== MODEL =====
model = ModelModule(num_labels=NUM_LABELS, model_name_or_path="microsoft/layoutlmv3-base")

state_dict = torch.load(MODEL_PATH, map_location="cpu")
model.load_state_dict(state_dict)

model.to(DEVICE)
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}

        logits, _ = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            bbox=batch["bbox"],
            pixel_values=batch["pixel_values"],
            labels=None
        )

        preds = torch.argmax(logits, dim=-1)

        preds = preds.view(-1).cpu().numpy()
        labels = batch["labels"].view(-1).cpu().numpy()

        mask = (labels != -100) & (labels != 0)

        all_preds.extend(preds[mask])
        all_labels.extend(labels[mask])

# ===== METRICS =====
accuracy = accuracy_score(all_labels, all_preds)
precision = precision_score(all_labels, all_preds, average="macro", zero_division=0)
recall = recall_score(all_labels, all_preds, average="macro", zero_division=0)
f1 = f1_score(all_labels, all_preds, average="macro", zero_division=0)

print("\n===== FINAL METRICS =====\n")
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

# ===== CLASSIFICATION REPORT =====
label_names = ["O", "HOSPITAL", "PATIENT", "DATE", "ADDRESS", "AMOUNT"]

print("\n===== CLASSIFICATION REPORT =====\n")
labels_list = [1, 2, 3, 4, 5]

label_names = ["HOSPITAL", "PATIENT", "DATE", "ADDRESS", "AMOUNT"]

print(classification_report(
    all_labels,
    all_preds,
    labels=labels_list,
    target_names=label_names,
    zero_division=0
))

Loading weights:   0%|          | 0/212 [00:00<?, ?it/s]

LayoutLMv3ForTokenClassification LOAD REPORT from: microsoft/layoutlmv3-base
Key                                | Status     | 
-----------------------------------+------------+-
layoutlmv3.embeddings.position_ids | UNEXPECTED | 
classifier.weight                  | MISSING    | 
classifier.bias                    | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



===== FINAL METRICS =====

Accuracy : 0.9812
Precision: 0.9872
Recall   : 0.9786
F1 Score : 0.9821

===== CLASSIFICATION REPORT =====

              precision    recall  f1-score   support

    HOSPITAL       1.00      1.00      1.00        35
     PATIENT       1.00      0.89      0.94        28
        DATE       1.00      1.00      1.00        25
     ADDRESS       0.94      1.00      0.97        44
      AMOUNT       1.00      1.00      1.00        28

    accuracy                           0.98       160
   macro avg       0.99      0.98      0.98       160
weighted avg       0.98      0.98      0.98       160

